# Kaggle candidate export — style-feature Random Forest
Uses the CV-selected configuration without further tuning. Fits on all 10,000 labeled rows, then predicts P(AI) for the 3,000 test rows. Does not upload to Kaggle. The earlier validation score belongs to the 8,000-row model, not this refit.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, hashlib, re, sys
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
SELECTION_DIR = PROJECT_DIR / "results"
selection = json.loads((SELECTION_DIR / "run_metadata.json").read_text())
assert selection["selected_representation"] == "style_only"
OUTPUT_DIR = PROJECT_DIR / "submissions" / datetime.now(timezone.utc).strftime("rf_style_%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


## Feature extraction
Exact StyleFeatures class copied from the executed improvement notebook, with no feature changes.

In [ ]:
from rf_features import StyleFeatures

## Refit and predict
All supervised labels come from train.csv. No test labels are available or used.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
template = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})
assert len(train) == 10000 and len(test) == len(template) == 3000
assert train["label"].notna().all() and set(train["label"]) == {0, 1}
assert test["id"].is_unique and template["id"].is_unique
assert set(test["id"]) == set(template["id"])
assert list(template.columns) == ["id", "label"]
train_hash = hashlib.sha256((DATA_DIR / "train.csv").read_bytes()).hexdigest()
assert train_hash == selection["dataset_sha256"]
transformer = StyleFeatures()
A = transformer.fit_transform(train["text"].fillna(""))
B = transformer.transform(test["text"].fillna(""))
assert A.shape == (10000, 21) and B.shape == (3000, 21)
model = RandomForestClassifier(**selection["selected_params"])
model.fit(A, train["label"].to_numpy())
probabilities = model.predict_proba(B)[:, list(model.classes_).index(1)]
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
probability_by_id = pd.Series(probabilities, index=test["id"])
submission = template.copy()
submission["label"] = submission["id"].map(probability_by_id)
assert submission["label"].notna().all()
assert submission["id"].tolist() == template["id"].tolist()
submission_path = OUTPUT_DIR / "submission_rf_style.csv"
submission.to_csv(submission_path, index=False)
check = pd.read_csv(submission_path, dtype={"id": str})
assert list(check.columns) == ["id", "label"] and len(check) == 3000
assert check["id"].tolist() == template["id"].tolist()
assert np.allclose(check["label"], submission["label"], atol=1e-14, rtol=0)
metadata = {
    "selected_from": str(SELECTION_DIR), "training_rows": len(train),
    "test_rows": len(test), "features": 21, "rf_params": selection["selected_params"],
    "train_sha256": train_hash,
    "test_sha256": hashlib.sha256((DATA_DIR / "test.csv").read_bytes()).hexdigest(),
    "submission_sha256": hashlib.sha256(submission_path.read_bytes()).hexdigest(),
    "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
    "pandas_version": pd.__version__, "python_version": sys.version,
    "meaning": "label is P(AI=1)", "uploaded_to_kaggle": False,
    "note": "Earlier validation AUROC 0.989913 used an 8000-row training partition; no Kaggle score yet."
}
(OUTPUT_DIR / "submission_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Validated:", submission_path)
print("Rows:",len(check),"Probability range:",check.label.min(),check.label.max())
display(check.head())
